In [ ]:
"""
TODO:
    - probably give some statistics (frequency every X lines of code)
"""

'\nTODO:\n    - probably give some statistics (frequency every X lines of code)\n'

In [ ]:
# Python and ipynb API analyzer

import ast
import json
from collections import defaultdict
from pathlib import Path
from tempfile import NamedTemporaryFile


def _annotate_parents(tree: ast.AST) -> None:
    """Add `.parent` links so we can walk up the tree."""
    for parent in ast.walk(tree):
        for child in ast.iter_child_nodes(parent):
            child.parent = parent


def _in_type_checking_block(node: ast.AST) -> bool:
    """Check if in an `if TYPE_CHECKING:` block."""
    while hasattr(node, "parent"):
        parent = node.parent
        if isinstance(parent, ast.If):
            if isinstance(parent.test, ast.Name) and parent.test.id == "TYPE_CHECKING":
                return True
        node = parent
    return False


class ApiAnalyzerPy(ast.NodeVisitor):
    """API analyzer for .py file."""

    def __init__(self, package: str) -> None:
        self.package = package

        # alias map: local name → full path
        self.aliases: dict[str, str] = {}
        # usage counts
        self.usage: dict[str, int] = defaultdict(int)

    def visit_Import(self, node) -> None:
        if _in_type_checking_block(node):
            return
        for alias in node.names:
            if alias.name.startswith(self.package):
                asname = alias.asname or alias.name.split(".")[-1]
                self.aliases[asname] = alias.name

    def visit_ImportFrom(self, node) -> None:
        if _in_type_checking_block(node):
            return
        if node.module and node.module.startswith(self.package):
            for alias in node.names:
                asname = alias.asname or alias.name
                self.aliases[asname] = f"{node.module}.{alias.name}"

    def visit_Call(self, node) -> None:
        """Track function/method calls."""
        if isinstance(node.func, ast.Attribute):
            base = node.func.value
            if isinstance(base, ast.Name) and base.id in self.aliases:
                full = f"{self.aliases[base.id]}.{node.func.attr}"
                self.usage[full] += 1
        self.generic_visit(node)


def analyze_py(path: str | Path, package: str) -> tuple[dict[str, str], dict[str, int]]:
    """
    Analyze a Python file for package API usage.

    Returns:
        aliases (dict): Mapping of local names → full package paths.
        usage (dict): Mapping of package API calls → count.
    """
    path = Path(path)

    if path.suffix != ".py":
        raise ValueError(f"cannot analyze non-py file: {path}")

    text = path.read_text(encoding="utf-8")

    tree = ast.parse(text)

    _annotate_parents(tree)

    analyzer = ApiAnalyzerPy(package)
    analyzer.visit(tree)
    return analyzer.aliases, dict(analyzer.usage)


def analyze_notebook(
    path: str | Path, package: str
) -> tuple[dict[str, str], dict[str, int]]:
    def clean_notebook_code(code: str) -> str:
        cleaned: list[str] = []
        for line in code.splitlines():
            if not line or line.startswith(("!", "%", "?")):
                continue  # skip shell/magic/help commands
            cleaned.append(line)
        return "\n".join(cleaned)

    path = Path(path)
    if path.suffix != ".ipynb":
        raise ValueError(f"cannot analyze non-ipynb file: {path}")

    nb = json.loads(path.read_text(encoding="utf-8"))
    aliases, usage = {}, {}

    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        code = "".join(cell.get("source", []))
        code = clean_notebook_code(code)
        if not code.strip():
            continue

        # Write code to a temporary .py file
        with NamedTemporaryFile("w", suffix=".py", delete=False) as tmp:
            tmp.write(code)
            tmp_path = Path(tmp.name)

        _alias, _usage = analyze_py(tmp_path, package)

        # Merge results
        aliases.update(_alias)
        for key, val in _usage.items():
            usage[key] = usage.get(key, 0) + val

        tmp_path.unlink(missing_ok=False)

    return aliases, usage

In [ ]:
def analyze_dir(
    dir_path: str | Path, package: str
) -> tuple[dict[str, str], dict[str, int]]:
    """
    Recursively analyze all .py and .ipynb files in a directory
    for package API usage.

    Args:
        dir_path: directory path

    Returns:
        aliases (dict): merged local → full package paths
        usage (dict): merged API usage counts across all files
    """
    dir_path = Path(dir_path)
    if not dir_path.is_dir():
        raise NotADirectoryError(f"Not a directory: {dir_path}")

    all_aliases: dict[str, str] = {}
    all_usage: dict[str, int] = defaultdict(int)

    for path in dir_path.rglob("*"):
        if path.name.startswith("."):  # skip hidden files/dirs
            continue
        if path.suffix == ".py":
            try:
                aliases, usage = analyze_py(path, package)
            except Exception as e:
                print(f"⚠️ Skipping {path} (error: {e})")
                continue
        elif path.suffix == ".ipynb":
            try:
                aliases, usage = analyze_notebook(path, package)
            except Exception as e:
                print(f"⚠️ Skipping {path} (error: {e})")
                continue
        else:
            continue

        # merge aliases and usage
        all_aliases.update(aliases)
        for k, v in usage.items():
            all_usage[k] += v

    return all_aliases, dict(all_usage)

In [ ]:
"""Sanity check with .py and .ipynb files."""
test_py_file = "./dependent-repos/pymatviz/pymatviz/chem_env.py"
aliases, usage = analyze_py(test_py_file, "pymatgen")
print(aliases)
print(usage)

test_ipynb_file = "./dependent-repos/pymatviz/examples/widgets/jupyter_demo.ipynb"
aliases, usage = analyze_notebook(test_ipynb_file, "pymatgen")
print(aliases)
print(usage)

test_dir = "./dependent-repos/pymatviz/"
aliases, usage = analyze_dir(test_dir, "pymatgen")
print(aliases)
print(usage)

{'local_env': 'pymatgen.analysis.local_env'}
{'pymatgen.analysis.local_env.LocalStructOrderParams': 1, 'pymatgen.analysis.local_env.CrystalNN': 1}
{'Composition': 'pymatgen.core.Composition', 'Lattice': 'pymatgen.core.Lattice', 'Structure': 'pymatgen.core.Structure'}
{'pymatgen.core.Lattice.cubic': 1}
{'Lattice': 'pymatgen.core.Lattice', 'Structure': 'pymatgen.core.Structure', 'AseAtomsAdaptor': 'pymatgen.io.ase.AseAtomsAdaptor', 'PhononBands': 'pymatgen.phonon.bandstructure.PhononBandStructureSymmLine', 'PhononDos': 'pymatgen.phonon.dos.PhononDos', 'Composition': 'pymatgen.core.Composition', 'IStructure': 'pymatgen.core.IStructure', 'DiffractionPattern': 'pymatgen.analysis.diffraction.xrd.DiffractionPattern', 'XRDCalculator': 'pymatgen.analysis.diffraction.xrd.XRDCalculator', 'MPRester': 'pymatgen.ext.matproj.MPRester', 'local_env': 'pymatgen.analysis.local_env', 'SiteCollection': 'pymatgen.core.SiteCollection', 'get_pmg_structure': 'pymatgen.io.phonopy.get_pmg_structure', 'MSONAtoms'

In [ ]:
# Clone top X repos
import pandas as pd
import subprocess
from pathlib import Path

INPUT_FILE: str = "pymatgen_dependents_with_stars.csv"
OUTPUT_DIR: Path = Path("dependent-repos")
TOP_N: int = 5  # TODO: try with 5 first

OUTPUT_DIR.mkdir(exist_ok=True)

repos_df: pd.DataFrame = pd.read_csv(INPUT_FILE).head(TOP_N)

for _, row in repos_df.iterrows():
    repo_url: str = row["repo_url"]
    repo_name: str = Path(repo_url).name
    target_path: str = OUTPUT_DIR / repo_name

    print(f"Processing {repo_name}...")

    try:
        subprocess.run(
            ["git", "clone", "--depth=1", repo_url],
            cwd=OUTPUT_DIR,
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"✓ Cloned {repo_name}")
    except subprocess.CalledProcessError as e:
        # git clone returns 128 if the folder exists, but we ignore it
        if "already exists" in e.stderr:
            print(f"⚠ Skipping {repo_name}, already cloned")
        else:
            print(f"✗ Failed cloning {repo_name}: {e.stderr.strip()}")

Processing materials_discovery...
✓ Cloned materials_discovery
Processing Uni-Mol...
✓ Cloned Uni-Mol
Processing AIRS...
✓ Cloned AIRS
Processing mat2vec...
✓ Cloned mat2vec
Processing megnet...
✓ Cloned megnet


In [ ]:
# Analyze top X repos
# Need to manually build a glob mapping for each repo

# TODO: or perhaps just exclude `tests/.*/...`?

REPO_PATH_MAP: dict[str, list[str]] = {
    "materials_discovery": ["notebooks"],
    "Uni-Mol": ["unimol*"],
    # "AIRS": ["airs*"],
    "mat2vec": ["mat2vec"],
    "megnet": ["megnet"],
}